In [ ]:
import sys
import os
from dotenv import load_dotenv
load_dotenv()

sys.path.append("..")

from pytorch_lightning.loggers import CometLogger
from pathlib import Path
from datetime import datetime, timedelta
from lib.wpt_utils import load_wavelets
from training.logging_utils import log_metrics
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl

In [ ]:
COMET_API_KEY = os.environ.get("COMET_API_KEY")

In [ ]:
SEQ_1_PATH = Path("../data/wpts/sequence_1_db35")
SEQ_2_PATH = Path("../data/wpts/sequence_2_db35")
SEQ_3_PATH = Path("../data/wpts/sequence_3_db35")

In [ ]:
data_seq_1 = load_wavelets(SEQ_1_PATH)
data_seq_2 = load_wavelets(SEQ_2_PATH)
data_test = load_wavelets(SEQ_3_PATH)

In [ ]:
from training.training_pipeline import run_pipeline
from training.objects import PipelineOutput

In [ ]:
data_train, data_valid = data_seq_2, data_seq_1

In [ ]:
import numpy as np

# Single

In [ ]:
params = {
    "normalize": True,
    "batch_size": 128,
    "num_epochs": 400,
    "lr": 1e-3,
    "hidden_dim": 16,
    "num_layers": 3,
    "activation": "relu",
    "fs": 256,
    "bandwidth": 113,
    "beta": 0.001,
    "f_min": 7,
    "kl_weight": 0.00001,
    "latent_dim": 4,
    "max_level": 6,
    "min_level": 3,
}

In [ ]:
logger = CometLogger(
    api_key=COMET_API_KEY,
project_name=os.environ.get("COMET_PROJECT_VAE"),
workspace=os.environ.get("COMET_WORKSPACE"),
)

In [ ]:
logger.log_hyperparams(params)

In [ ]:
output, train_results, valid_results, test_results = run_pipeline(
    data_train=data_train,
    data_valid=data_valid,
    params=params,
    logger=logger,
    return_model=True,
    data_test=data_test,
)

In [ ]:
cable = 1
mu = np.array(output.train_mu)
std = np.array(output.train_std)

y = torch.sigmoid(torch.tensor(y))
z = train_results.z[:,cable,:]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.pcolormesh(x)
plt.subplot(1,2,2)
plt.pcolormesh(y)

plt.figure(figsize=(10,4))
plt.plot(z)

In [ ]:
log_metrics(output, logger)

In [ ]:
logger.experiment.end()

In [ ]:
df_train = pd.DataFrame([o.model_dump() for o in output.train_vae_results])
df_valid = pd.DataFrame([o.model_dump() for o in output.valid_vae_results])
df_test = pd.DataFrame([o.model_dump() for o in output.test_vae_results])

In [ ]:
def extract_arrays(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, list[datetime]]:
    x = []
    t = []
    for cable in [0, 1, 2, 3]:
        df_cable = df.loc[df.cable == cable].copy()
        df_cable.sort_values("sampled_at")
        x.append(np.array(df_cable["values"].to_list()))
        sampled_at = df_cable["sampled_at"].to_list()
        t.append(np.array(df_cable["tension"].to_list()))
    x = np.array(x)
    t = np.array(t)
    return x, t, sampled_at

In [ ]:
x_train, t_train, s_train = extract_arrays(df_train)
x_valid, t_valid, s_valid = extract_arrays(df_valid)
x_test, t_test, s_test = extract_arrays(df_test)

In [ ]:
# n_latent = x_train.shape[2]
# n_cables = x_train.shape[0]

In [ ]:
dist_train = pd.DataFrame([o.model_dump() for o in output.train_dist_results])
dist_valid = pd.DataFrame([o.model_dump() for o in output.valid_dist_results])
dist_test = pd.DataFrame([o.model_dump() for o in output.test_dist_results])

In [ ]:
d_train, t_train, _ = extract_arrays(dist_train)
d_valid, t_valid, _ = extract_arrays(dist_valid)
d_test, t_test, _ = extract_arrays(dist_test)

d_train = d_train.squeeze()
d_valid = d_valid.squeeze()
d_test = d_test.squeeze()

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(4):
    plt.subplot(2, 2, i + 1)
    plt.plot(t_train[i, :], d_train[i, :], label=f"Train Cable {i}")
    plt.plot(t_valid[i, :], d_valid[i, :], label=f"Valid Cable {i}")
    plt.plot(t_test[i, :], d_test[i, :], label=f"Test Cable {i}")

In [ ]:
plt.figure(figsize=(12, 8))
for i in range(4):
    # plt.subplot(2,2,i+1)
    plt.plot(t_train[i, :], d_train[i, :], label=f"Train Cable {i}")
    plt.plot(t_valid[i, :], d_valid[i, :], label=f"Valid Cable {i}")
    plt.plot(t_test[i, :], d_test[i, :], label=f"Test Cable {i}")

In [ ]:
plt.figure(figsize=(12, 12))
for j in range(4):
    plt.subplot(2, 2, j + 1)
    for i in range(4):
        plt.plot(t_train[i, :], x_train[i, :, j], label=f"Train Cable {i}")
        plt.plot(t_valid[i, :], x_valid[i, :, j], label=f"Valid Cable {i}")
        plt.plot(t_test[i, :], x_test[i, :, j], label=f"Test Cable {i}")

# Save data

In [ ]:
import json
from pydantic import BaseModel

In [ ]:
path_train = Path("../data/embeddings/sequence_train")
path_valid = Path("../data/embeddings/sequence_valid")
path_test = Path("../data/embeddings/sequence_test")

In [ ]:
x_train.shape, t_train.shape, len(s_train)

In [ ]:
class Embedding(BaseModel):
    cable: int
    tension: float
    values: list[float]

In [ ]:
class SampleEmbeddings(BaseModel):
    sampled_at: datetime
    c0: Embedding
    c1: Embedding
    c2: Embedding
    c3: Embedding

In [ ]:
for k in range(len(s_train)):
    sample = SampleEmbeddings(
        sampled_at=s_train[k],
        c0=Embedding(
            cable=0, tension=float(t_train[0, k]), values=x_train[0, k, :].tolist()
        ),
        c1=Embedding(
            cable=1, tension=float(t_train[1, k]), values=x_train[1, k, :].tolist()
        ),
        c2=Embedding(
            cable=2, tension=float(t_train[2, k]), values=x_train[2, k, :].tolist()
        ),
        c3=Embedding(
            cable=3, tension=float(t_train[3, k]), values=x_train[3, k, :].tolist()
        ),
    )
    filename = s_train[k].isoformat().replace(":", "-")
    data = sample.model_dump()
    data["sampled_at"] = data["sampled_at"].isoformat()
    with open(path_train / f"{filename}.json", "w") as f:
        json.dump(data, f, indent=4)

In [ ]:
for k in range(len(s_valid)):
    sample = SampleEmbeddings(
        sampled_at=s_valid[k],
        c0=Embedding(
            cable=0, tension=float(t_valid[0, k]), values=x_valid[0, k, :].tolist()
        ),
        c1=Embedding(
            cable=1, tension=float(t_valid[1, k]), values=x_valid[1, k, :].tolist()
        ),
        c2=Embedding(
            cable=2, tension=float(t_valid[2, k]), values=x_valid[2, k, :].tolist()
        ),
        c3=Embedding(
            cable=3, tension=float(t_valid[3, k]), values=x_valid[3, k, :].tolist()
        ),
    )
    filename = s_valid[k].isoformat().replace(":", "-")
    data = sample.model_dump()
    data["sampled_at"] = data["sampled_at"].isoformat()
    with open(path_valid / f"{filename}.json", "w") as f:
        json.dump(data, f, indent=4)

In [ ]:
for k in range(len(s_test)):
    sample = SampleEmbeddings(
        sampled_at=s_test[k],
        c0=Embedding(
            cable=0, tension=float(t_test[0, k]), values=x_test[0, k, :].tolist()
        ),
        c1=Embedding(
            cable=1, tension=float(t_test[1, k]), values=x_test[1, k, :].tolist()
        ),
        c2=Embedding(
            cable=2, tension=float(t_test[2, k]), values=x_test[2, k, :].tolist()
        ),
        c3=Embedding(
            cable=3, tension=float(t_test[3, k]), values=x_test[3, k, :].tolist()
        ),
    )
    filename = s_test[k].isoformat().replace(":", "-")
    data = sample.model_dump()
    data["sampled_at"] = data["sampled_at"].isoformat()
    with open(path_test / f"{filename}.json", "w") as f:
        json.dump(data, f, indent=4)

# Remove later

## Model

In [ ]:
from pytorch_lightning.loggers import CometLogger
import torch
import pytorch_lightning as pl
import os
from dotenv import load_dotenv
load_dotenv()

COMET_API_KEY = os.environ.get("COMET_API_KEY")


def get_activation(activation_fn: str) -> torch.nn.Module:
    """
    Get the activation function based on the input string.

    Args:
        activation_fn (str): The activation function name. Valid options are "relu", "leakyrelu", "sigmoid", "tanh".

    Returns:
        torch.nn.Module: The corresponding activation function.
    """
    activation_dict = {
        "relu": torch.nn.ReLU,
        "leakyrelu": torch.nn.LeakyReLU,
        "sigmoid": torch.nn.Sigmoid,
        "tanh": torch.nn.Tanh,
        "relu6": torch.nn.ReLU6,
        "gelu": torch.nn.GELU,
        "elu": torch.nn.ELU,
        "selu": torch.nn.SELU,
        "softmax": torch.nn.Softmax,
        "hardtanh": torch.nn.Hardtanh,
        "tanhshrink": torch.nn.Tanhshrink,
    }

    # Convert to lowercase and check the activation function dictionary
    activation_fn = activation_fn.lower()

    if activation_fn in activation_dict:
        return activation_dict[activation_fn]
    else:
        raise ValueError(f"Unsupported activation function: {activation_fn}")


class MLPEncoder(torch.nn.Module):
    """
    Multi-Layer Perceptron (MLP) encoder for the autoencoder.

    Args:
        input_dim (int): Dimension of the input data.
        embedding_dim (int): Dimension of the encoded embedding.
        hidden_dim (int): Number of neurons in the hidden layers.
        num_layers (int): Number of hidden layers in the encoder.
    """

    def __init__(
        self,
        input_dim: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int,
        activation_fn: str = "relu",
    ) -> None:
        super().__init__()
        self.input_dim: int = input_dim
        self.embedding_dim: int = embedding_dim
        self.hidden_dim: int = hidden_dim
        self.activation_fn: str = activation_fn

        # Get the activation function based on the string
        activation = get_activation(activation_fn)

        layers: list[torch.nn.Module] = [
            torch.nn.Linear(input_dim, hidden_dim),
            activation(),
        ]
        for _ in range(num_layers):
            layers += [
                torch.nn.Linear(hidden_dim, hidden_dim),
                activation(),
            ]
        layers.append(torch.nn.Linear(hidden_dim, embedding_dim))
        self.model: torch.nn.Sequential = torch.nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the encoder.

        Args:
            x (torch.Tensor): Input tensor of shape `(batch_size, input_dim)`.

        Returns:
            torch.Tensor: Encoded tensor of shape `(batch_size, embedding_dim)`.
        """
        return self.model(x)


class MLPDecoder(torch.nn.Module):
    """
    Multi-Layer Perceptron (MLP) decoder for the autoencoder.

    Args:
        input_dim (int): Dimension of the input data to be reconstructed.
        embedding_dim (int): Dimension of the encoded embedding.
        hidden_dim (int): Number of neurons in the hidden layers.
        num_layers (int): Number of hidden layers in the decoder.
    """

    def __init__(
        self,
        output_dim: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int,
        activation_fn: str = "relu",
    ) -> None:
        super().__init__()
        self.output_dim: int = output_dim
        self.embedding_dim: int = embedding_dim
        self.hidden_dim: int = hidden_dim
        self.activation_fn: str = activation_fn

        # Get the activation function based on the string
        activation = get_activation(activation_fn)

        layers: list[torch.nn.Module] = [
            torch.nn.Linear(embedding_dim, hidden_dim),
            activation(),
        ]
        for _ in range(num_layers):
            layers += [
                torch.nn.Linear(hidden_dim, hidden_dim),
                activation(),
            ]
        layers += [torch.nn.Linear(hidden_dim, output_dim)]
        self.model: torch.nn.Sequential = torch.nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the decoder.

        Args:
            x (torch.Tensor): Encoded tensor of shape `(batch_size, embedding_dim)`.

        Returns:
            torch.Tensor: Reconstructed tensor of shape `(batch_size, input_dim)`.
        """
        y = self.model(x)
        return y


class VAERegularized(pl.LightningModule):
    def __init__(
        self,
        input_dim: int,
        lr: float,
        latent_dim: int,
        hidden_dim: int,
        num_layers: int,
        activation: str,
        beta: float,
        kl_weight: float = 1.0,
    ):
        """
        A LightningModule that bundles together:
          - An encoder/decoder for variational autoencoder
          - Regularization in terms of cable embedding distance
          - Loss functions for reconstruction, KL divergence, and distance
        """
        super().__init__()

        # Encoder outputs both mu and logvar
        self.encoder_mu = MLPEncoder(
            input_dim=input_dim,
            embedding_dim=latent_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            activation_fn=activation,
        )
        self.encoder_logvar = MLPEncoder(
            input_dim=input_dim,
            embedding_dim=latent_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            activation_fn=activation,
        )

        self.decoder = MLPDecoder(
            output_dim=input_dim,
            embedding_dim=latent_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            activation_fn=activation,
        )
        self.dropout = torch.nn.Dropout(p=0.2)

        # Store loss functions, hyperparameters
        # self.rec_loss_fn = torch.nn.MSELoss(reduction="mean")
        self.rec_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="mean")
        self.lr = lr
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.activation = activation
        self.beta = beta
        self.kl_weight = kl_weight

    def encode(self, x: torch.Tensor):
        """
        Encode input to latent distribution parameters.

        Args:
            x (torch.Tensor): Input tensor

        Returns:
            tuple: (mu, logvar) - mean and log variance of latent distribution
        """
        x_drop = self.dropout(x)
        mu = self.encoder_mu(x_drop)
        logvar = self.encoder_logvar(x_drop)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        """
        Reparameterization trick to sample from latent distribution.

        Args:
            mu (torch.Tensor): Mean of latent distribution
            logvar (torch.Tensor): Log variance of latent distribution

        Returns:
            torch.Tensor: Sampled latent vector
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def kl_divergence(self, mu: torch.Tensor, logvar: torch.Tensor):
        """
        Compute KL divergence between latent distribution and standard normal.

        Args:
            mu (torch.Tensor): Mean of latent distribution
            logvar (torch.Tensor): Log variance of latent distribution

        Returns:
            torch.Tensor: KL divergence loss
        """
        return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    def forward(self, x: torch.Tensor):
        """
        Forward pass for inference:
         1) encode input into (mu, logvar)
         2) reparameterize to latent z
         3) decode z -> reconstruction
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_rec = self.decoder(z)
        return x_rec, None, z, None

    def training_step(self, batch, batch_idx):
        """
        Runs each optimization step:
         - forward pass
         - compute combined loss (reconstruction + KL + distance)
         - log metrics
         - return loss for backprop
        """
        x = batch.float()
        batch_size = x.shape[0]
        x_rec_list = []
        z_list = []
        rec_loss = 0.0
        kl_loss = 0.0
        num_cables = x.shape[1]

        for i in range(num_cables):
            mu, logvar = self.encode(x[:, i])
            z = self.reparameterize(mu, logvar)
            x_rec = self.decoder(z)

            rec_loss += self.rec_loss_fn(x_rec, x[:, i])
            kl_loss += self.kl_divergence(mu, logvar)
            x_rec_list.append(x_rec)
            z_list.append(z)

        # Compute the pairwise distance between the embeddings
        z_stack = torch.stack(z_list, dim=0).transpose(0, 1)
        z_dist = torch.cdist(z_stack, z_stack, p=2)
        upper_triangular = torch.triu(z_dist, diagonal=1)
        dist_loss = upper_triangular.sum() / (batch_size + num_cables)

        # Average KL loss over batch and cables
        kl_loss = kl_loss / (batch_size * num_cables)

        loss = rec_loss + self.kl_weight * kl_loss + self.beta * dist_loss

        # Log for visualization in Lightning
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.log("train_rec_loss", rec_loss, on_step=False, on_epoch=True)
        self.log("train_kl_loss", kl_loss, on_step=False, on_epoch=True)
        self.log("train_dist_loss", dist_loss, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        """
        Validation step for the VAE+regularization:
        - Forward pass
        - Compute losses (reconstruction, KL, distance)
        - Log metrics
        - No weight updates (handled automatically by Lightning)
        """
        x = batch.float()
        batch_size = x.shape[0]
        x_rec_list = []
        z_list = []
        rec_loss = 0.0
        kl_loss = 0.0
        num_cables = x.shape[1]

        for i in range(num_cables):
            mu, logvar = self.encode(x[:, i])
            z = self.reparameterize(mu, logvar)
            x_rec = self.decoder(z)

            rec_loss += self.rec_loss_fn(x_rec, x[:, i])
            kl_loss += self.kl_divergence(mu, logvar)
            x_rec_list.append(x_rec)
            z_list.append(z)

        # Compute the pairwise distance between the embeddings
        z_stack = torch.stack(z_list, dim=0).transpose(0, 1)
        z_dist = torch.cdist(z_stack, z_stack, p=2)
        upper_triangular = torch.triu(z_dist, diagonal=1)
        dist_loss = upper_triangular.sum() / (batch_size + num_cables)

        # Average KL loss over batch and cables
        kl_loss = kl_loss / (batch_size * num_cables)

        loss = rec_loss + self.kl_weight * kl_loss + self.beta * dist_loss

        # Log validation metrics
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_rec_loss", rec_loss, on_step=False, on_epoch=True)
        self.log("val_kl_loss", kl_loss, on_step=False, on_epoch=True)
        self.log("val_dist_loss", dist_loss, on_step=False, on_epoch=True)

        return {"val_loss": loss}

    def configure_optimizers(self):
        """
        Set up optimizers/schedulers for training.
        """
        optimizer = torch.optim.Adam(
            list(self.encoder_mu.parameters())
            + list(self.encoder_logvar.parameters())
            + list(self.decoder.parameters()),
            lr=self.lr,
        )
        return optimizer



## Runner

In [ ]:
import sys
from pytorch_lightning.loggers import CometLogger

sys.path.append(".")
sys.path.append("..")

from pydantic import BaseModel
import numpy as np
import pandas as pd
from classes.sample.sample_classes import SampleWaveletPacket
from training.objects import (
    PipelineOutput,
    VAEResults,
)
from torch.utils.data import DataLoader
# from training.models.ae_regularized import VAERegularized
import pytorch_lightning as pl
from training.evaluate import evaluate_vae
from training.objects import BandRange, Embedding
from datetime import timedelta
from copy import deepcopy
from datetime import datetime
from lib.wpt_utils import combine_wpts, build_index_per_level
from typing import Optional
from pytorch_lightning.callbacks import ModelCheckpoint


def extract_feature_vectors(
    wpt_dataset: list[SampleWaveletPacket],
) -> tuple[np.ndarray, np.ndarray, list[datetime]]:
    cables = [1, 2, 3, 4]
    x_per_cable: dict[int, list[float]] = {cable: [] for cable in cables}
    t_per_cable: dict[int, list[float]] = {cable: [] for cable in cables}
    dates: list[datetime] = []
    for d in wpt_dataset:
        for c in cables:
            cable_data = getattr(d, f"sample_c{c}")
            combined = combine_wpts([cable_data.y, cable_data.z])
            x_per_cable[c].append(combined.energy_map)
            t_per_cable[c].append(cable_data.tension)
        dates.append(datetime.fromisoformat(d.sample_id))

    x_tensor = np.array([np.array(x) for x in x_per_cable.values()]).swapaxes(0, 1)
    t_tensor = np.array([np.array(t) for t in t_per_cable.values()]).swapaxes(0, 1)
    return x_tensor, t_tensor, dates


def normalize_by_max_per_level(x_tensor: np.ndarray, max_level: int) -> np.ndarray:
    num_cables = x_tensor.shape[1]
    x_copy = deepcopy(x_tensor)
    index_per_level = build_index_per_level(0, max_level)
    for c in range(num_cables):
        for _, ind in index_per_level.items():
            level_values = x_tensor[:, c][:, ind]
            level_max = np.max(level_values, axis=1)
            x_copy[:, c, ind] = x_tensor[:, c, ind] / level_max.reshape(-1, 1)
    return x_copy


def get_valid_index(
    f_min: float, f_max: float, min_level: int, max_level: int, fs: float
):
    valid_index = {}
    indexes = build_index_per_level(min_level, max_level)
    for level, ind in indexes.items():
        delta_f = (fs / 2) / (2**level)
        valid_index[level] = []
        for k, i in enumerate(ind):
            freq_range = BandRange(start=k * delta_f, end=(k + 1) * delta_f)
            if freq_range.has_intersection(BandRange(start=f_min, end=f_max)):
                valid_index[level].append(i)
    return valid_index


def run_pipeline(
    data_train: list[SampleWaveletPacket],
    data_valid: list[SampleWaveletPacket],
    params: dict,
    logger: pl.loggers.Logger,
    return_model: bool = False,
    data_test: Optional[list[SampleWaveletPacket]] = None,
) -> PipelineOutput:
    # -- Get initial parameters --
    f_max = params["f_min"] + params["bandwidth"]
    max_level = data_train[0].sample_c1.y.max_level
    eps = 1e-4

    # -- Extract feature vectors --
    train_x_tensor, train_t_tensor, train_dates = extract_feature_vectors(data_train)
    valid_x_tensor, valid_t_tensor, valid_dates = extract_feature_vectors(data_valid)
    if data_test is not None:
        test_x_tensor, test_t_tensor, test_dates = extract_feature_vectors(data_test)

    # -- Normalize --
    # 1) max per band
    train_xn = normalize_by_max_per_level(train_x_tensor, max_level=max_level + 1)
    valid_xn = normalize_by_max_per_level(valid_x_tensor, max_level=max_level + 1)
    if data_test is not None:
        test_xn = normalize_by_max_per_level(test_x_tensor, max_level=max_level + 1)

    # 2) log
    # train_xn = np.log(train_xn)
    # valid_xn = np.log(valid_xn)
    # if data_test is not None:
    #     test_xn = np.log(test_xn)

    # 3) z-score
    train_mu = train_xn.reshape(-1, train_xn.shape[-1]).mean(axis=0)
    train_std = train_xn.reshape(-1, train_xn.shape[-1]).std(axis=0)

    # train_xn = (train_xn - train_mu) / (train_std + eps)
    # valid_xn = (valid_xn - train_mu) / (train_std + eps)
    # if data_test is not None:
    #     test_xn = (test_xn - train_mu) / (train_std + eps)

    # -- Select frequency bands --
    valid_index = get_valid_index(
        f_min=params["f_min"],
        f_max=f_max,
        min_level=params["min_level"],
        max_level=params["max_level"],
        fs=params["fs"],
    )
    valid_index_flat = [item for sublist in valid_index.values() for item in sublist]
    temp_train = np.zeros((train_xn.shape[0], train_xn.shape[1], len(valid_index_flat)))
    temp_valid = np.zeros((valid_xn.shape[0], valid_xn.shape[1], len(valid_index_flat)))
    if data_test is not None:
        temp_test = np.zeros(
            (test_xn.shape[0], test_xn.shape[1], len(valid_index_flat))
        )

    train_mu = train_mu[valid_index_flat]
    train_std = train_std[valid_index_flat]

    for cable in [0, 1, 2, 3]:
        temp_train[:, cable] = train_xn[:, cable][:, valid_index_flat]
        temp_valid[:, cable] = valid_xn[:, cable][:, valid_index_flat]
        if data_test is not None:
            temp_test[:, cable] = test_xn[:, cable][:, valid_index_flat]
    train_xn, valid_xn = temp_train, temp_valid
    if data_test is not None:
        test_xn = temp_test

    # -- Dataloaders --
    train_dataloader = DataLoader(
        train_xn, batch_size=params["batch_size"], shuffle=True, num_workers=0
    )
    train_dataloader_no_shuffle = DataLoader(
        train_xn, batch_size=params["batch_size"], shuffle=False, num_workers=0
    )
    valid_dataloader = DataLoader(
        valid_xn, batch_size=params["batch_size"], shuffle=False, num_workers=0
    )
    if data_test is not None:
        test_dataloader = DataLoader(
            test_xn, batch_size=params["batch_size"], shuffle=False, num_workers=0
        )

    # -- VAE Model --

    checkpoint_callback = ModelCheckpoint(
        dirpath="checkpoints/",
        filename="vae_regularized",
        save_top_k=1,
        save_last=True,
        monitor="val_loss",
        mode="min",
    )

    ae_model = VAERegularized(
        input_dim=train_xn.shape[2],
        lr=params["lr"],
        latent_dim=params["latent_dim"],
        hidden_dim=params["hidden_dim"],
        num_layers=params["num_layers"],
        activation=params["activation"],
        beta=params["beta"],
        kl_weight=params["kl_weight"],
    )
    trainer = pl.Trainer(
        logger=logger,
        max_epochs=params["num_epochs"],
        accelerator="gpu",
        enable_progress_bar=False,
        enable_model_summary=False,
        callbacks=[checkpoint_callback],
    )
    trainer.fit(
        ae_model, train_dataloaders=train_dataloader, val_dataloaders=valid_dataloader
    )

    ae_model.eval()
    model_path = checkpoint_callback.best_model_path
    logger.experiment.log_model("vae_regularized", model_path)

    # -- Get embedding features --
    train_x, train_z, train_mse, train_rec = evaluate_vae(
        train_dataloader_no_shuffle, ae_model
    )
    valid_x, valid_z, valid_mse, valid_rec = evaluate_vae(valid_dataloader, ae_model)
    if data_test is not None:
        test_x, test_z, test_mse, test_rec = evaluate_vae(test_dataloader, ae_model)

    train_results = VAEResults(
        x=train_x,
        z=train_z,
        mse=train_mse,
        rec=train_rec,
        tensions=train_t_tensor,
        dates=train_dates,
    )

    valid_results = VAEResults(
        x=valid_x,
        z=valid_z,
        mse=valid_mse,
        rec=valid_rec,
        tensions=valid_t_tensor,
        dates=valid_dates,
    )

    test_results = None
    if data_test is not None:
        test_results = VAEResults(
            x=test_x,
            z=test_z,
            mse=test_mse,
            rec=test_rec,
            tensions=test_t_tensor,
            dates=test_dates,
        )

    train_embeddings = []
    for z, t, d in zip(train_z, train_t_tensor, train_dates):
        for cable in [0, 1, 2, 3]:
            train_embeddings.append(
                Embedding(
                    cable=cable,
                    sampled_at=d,
                    tension=t[cable],
                    values=z[cable].tolist(),
                )
            )

    valid_embeddings = []
    for z, t, d in zip(valid_z, valid_t_tensor, valid_dates):
        for cable in [0, 1, 2, 3]:
            valid_embeddings.append(
                Embedding(
                    cable=cable,
                    sampled_at=d,
                    tension=t[cable],
                    values=z[cable].tolist(),
                )
            )

    if data_test is not None:
        test_embeddings = []
        for z, t, d in zip(test_z, test_t_tensor, test_dates):
            for cable in [0, 1, 2, 3]:
                test_embeddings.append(
                    Embedding(
                        cable=cable,
                        sampled_at=d,
                        tension=t[cable],
                        values=z[cable].tolist(),
                    )
                )

    train_df = pd.DataFrame([f.model_dump() for f in train_embeddings])
    valid_df = pd.DataFrame([f.model_dump() for f in valid_embeddings])
    if data_test is not None:
        test_df = pd.DataFrame([f.model_dump() for f in test_embeddings])

    # -- Get baseline --
    train_start = train_df.sampled_at.min()
    train_end = train_start + timedelta(days=14)

    valid_start = valid_df.sampled_at.min()
    valid_end = valid_start + timedelta(days=14)

    if data_test is not None:
        test_start = test_df.sampled_at.min()
        test_end = test_start + timedelta(days=14)

    train_baseline_per_cable = (
        train_df.loc[train_df.sampled_at.isin(pd.date_range(train_start, train_end))]
        .groupby("cable")
        .agg({"values": lambda x: np.mean(np.vstack(x), axis=0).tolist()})
        .reset_index()
    )
    valid_baseline_per_cable = (
        valid_df.loc[valid_df.sampled_at.isin(pd.date_range(valid_start, valid_end))]
        .groupby("cable")
        .agg({"values": lambda x: np.mean(np.vstack(x), axis=0).tolist()})
        .reset_index()
    )
    if data_test is not None:
        test_baseline_per_cable = (
            test_df.loc[test_df.sampled_at.isin(pd.date_range(test_start, test_end))]
            .groupby("cable")
            .agg({"values": lambda x: np.mean(np.vstack(x), axis=0).tolist()})
            .reset_index()
        )

    # -- Compute distance per cable --
    centered_train_embeddings = []
    centered_valid_embeddings = []
    centered_test_embeddings = []

    for train_embedding in train_embeddings:
        cable = train_embedding.cable
        baseline = train_baseline_per_cable.loc[
            train_baseline_per_cable["cable"] == cable
        ]["values"].item()
        difference = np.array(train_embedding.values) - np.array(baseline)
        dist = np.linalg.norm(difference)
        centered_train_embeddings.append(
            Embedding(
                cable=train_embedding.cable,
                sampled_at=train_embedding.sampled_at,
                tension=train_embedding.tension,
                values=[dist],
            )
        )

    for valid_embedding in valid_embeddings:
        cable = valid_embedding.cable
        baseline = valid_baseline_per_cable.loc[
            valid_baseline_per_cable["cable"] == cable
        ]["values"].item()
        difference = np.array(valid_embedding.values) - np.array(baseline)
        dist = np.linalg.norm(difference)
        centered_valid_embeddings.append(
            Embedding(
                cable=valid_embedding.cable,
                sampled_at=valid_embedding.sampled_at,
                tension=valid_embedding.tension,
                values=[dist],
            )
        )

    if data_test is not None:
        for test_embedding in test_embeddings:
            cable = test_embedding.cable
            baseline = test_baseline_per_cable.loc[
                test_baseline_per_cable["cable"] == cable
            ]["values"].item()
            difference = np.array(test_embedding.values) - np.array(baseline)
            dist = np.linalg.norm(difference)
            centered_test_embeddings.append(
                Embedding(
                    cable=test_embedding.cable,
                    sampled_at=test_embedding.sampled_at,
                    tension=test_embedding.tension,
                    values=[dist],
                )
            )

    return (
        PipelineOutput(
            train_vae_results=train_embeddings,
            valid_vae_results=valid_embeddings,
            test_vae_results=test_embeddings if data_test is not None else None,
            train_dist_results=centered_train_embeddings,
            valid_dist_results=centered_valid_embeddings,
            test_dist_results=(
                centered_test_embeddings if data_test is not None else None
            ),
            model=ae_model if return_model else None,
            train_mu=train_mu.tolist(),
            train_std=train_std.tolist(),
        ),
        train_results,
        valid_results,
        test_results,
    )